# 03 Results analysis

This notebook answers one question: given the outputs of a completed training
run, did any model demonstrate an edge that survives being looked at per fold,
per asset, per regime and per seed.

It reads from `reports/results/` only. It does not train anything and it does not
download anything. If no completed run is present it says so and every later cell
degrades to a printed message rather than an exception, so the notebook can be
opened and read before the first run exists.

**Run this notebook from the repository root**, so that `from src...` imports
resolve. If it is launched from inside `notebooks/`, the first code cell walks
one directory up to find the repository root.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import schema
from src.config import load_config
from src.data.build_panel import load_panel
from src.data.targets import build_targets
from src.evaluate import report as rep

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# Set RUN_DIR to a specific directory under reports/results to analyse that run.
# Leaving it as None uses the most recently written completed run, selected by
# modification time, so the run analysed here changes as new runs are written.
# Pin it to a directory name when a result needs to stay reproducible.
RUN_DIR = None

RESULTS_ROOT = ROOT / "reports" / "results"
print("repository root:", ROOT)

## Loading a run

`latest_run` picks the most recently written directory under `reports/results`
that contains a `results.csv`. A run directory is named by the hash of the merged
config that produced it, so the run identifier below is enough to recover the
exact settings behind every number in this notebook.

In [ ]:
run_dir = None
results = pd.DataFrame()
predictions = pd.DataFrame()
load_error = None

try:
    run_dir = Path(RUN_DIR) if RUN_DIR is not None else rep.latest_run(RESULTS_ROOT)
    results, predictions = rep.load_run(run_dir)
except (FileNotFoundError, NotADirectoryError, OSError) as error:
    load_error = error

HAS_RUN = run_dir is not None and not results.empty

if not HAS_RUN:
    print("No completed run was found under %s." % RESULTS_ROOT)
    print("Run `make train` from the repository root first, then re-run this notebook.")
    if load_error is not None:
        print("Loader reported: %s" % load_error)
else:
    print("run directory: %s" % run_dir)
    print("rows in results.csv: %d" % len(results))
    print("models: %s" % sorted(results["model"].unique()))
    print("folds: %d" % results["fold"].nunique())
    print("prediction rows: %d" % len(predictions))

## What kind of run is this

The pipeline can train a classification target (`dir_1d`) or a regression target
(`ret_1d`, `vol_1d`), and the metrics written to `results.csv` differ between
them. The cell below reads the run rather than assuming, and fixes three
variables that the rest of the notebook depends on: the headline metric, whether
lower is better for it, and which model serves as the baseline.

The distinction matters for reading every table below. For an accuracy style
metric a positive difference against the baseline is an improvement. For an error
style metric such as RMSE the sign is reversed, and that reversal is called out
again where it appears.

In [ ]:
TARGET = None
IS_CLASSIFICATION = False
METRIC = None
LOWER_IS_BETTER = False
BASELINE = None

BASELINE_PREFERENCE = [
    "persistence",
    "vol_persistence",
    "majority_class",
    "historical_mean",
    "zero",
]

if HAS_RUN:
    TARGET = str(results["target"].dropna().iloc[0])
    IS_CLASSIFICATION = "directional_accuracy" in results.columns

    if IS_CLASSIFICATION:
        METRIC = "directional_accuracy"
    else:
        METRIC = next((c for c in ("rmse", "mae", "r2_oos") if c in results.columns), None)

    LOWER_IS_BETTER = METRIC in {"rmse", "mae", "brier_score"}

    models = list(results["model"].unique())
    BASELINE = next((m for m in BASELINE_PREFERENCE if m in models), models[0] if models else None)

    print("target:              %s" % TARGET)
    print("task:                %s" % ("classification" if IS_CLASSIFICATION else "regression"))
    print("headline metric:     %s" % METRIC)
    print("direction:           %s" % ("lower is better" if LOWER_IS_BETTER else "higher is better"))
    print("baseline model:      %s" % BASELINE)
else:
    print("No run loaded, nothing to characterise.")

## Per fold summary

`fold_table` reports the mean and the standard deviation of each metric across
folds, one row per model. The standard deviation is the column that decides
whether a comparison means anything. Where the gap between two models is smaller
than the fold to fold spread of either of them, this experiment has not
distinguished them, however the ordering looks.

In [ ]:
summary_table = pd.DataFrame()
if HAS_RUN:
    summary_table = rep.fold_table(results)
    display(summary_table)
else:
    print("No run loaded, per fold summary unavailable.")

## The headline metric over time

Averages hide the thing most worth knowing, which is whether performance is
stable. A model averaging the same score as the baseline while swinging widely
around it across folds is a different object from one tracking the baseline
closely, and only a per fold plot separates them.

The baseline is drawn as a dashed line. For a classification run a horizontal
reference at 0.5 is added, but note that 0.5 is not the honest hurdle: the base
rate of the direction target moves fold by fold, so the baseline series itself is
the comparison that counts.

In [ ]:
if HAS_RUN and METRIC is not None:
    wide = results.pivot_table(
        index="test_start", columns="model", values=METRIC, aggfunc="mean"
    ).sort_index()
    wide.index = pd.to_datetime(wide.index)

    figure, axis = plt.subplots(figsize=(12, 5.5))
    for column in wide.columns:
        is_baseline = column == BASELINE
        axis.plot(
            wide.index,
            wide[column],
            "--" if is_baseline else "-",
            linewidth=2.2 if is_baseline else 1.3,
            marker="o",
            markersize=3,
            label=column + (" (baseline)" if is_baseline else ""),
        )
    if IS_CLASSIFICATION:
        axis.axhline(0.5, color="black", linewidth=0.8, alpha=0.6)
    axis.set_xlabel("test window start")
    axis.set_ylabel(METRIC.replace("_", " "))
    axis.set_title("%s per walk forward fold" % METRIC.replace("_", " "))
    axis.legend(fontsize=8, ncol=3)
    axis.tick_params(axis="x", rotation=45)
    figure.tight_layout()
    plt.show()

    display(wide.round(4))
else:
    print("No run loaded, per fold plot unavailable.")

## Edge over the baseline

`edge_over_baseline` differences each model against the baseline within the same
fold before averaging. Differencing inside the fold removes the fold's own
difficulty from the comparison, which a difference of two averages does not.

`win_rate` is the share of folds in which the difference is positive. It is a
distribution free read that does not depend on the size of any single fold's
result, and it is often the more honest number: a model ahead on half the folds
has not established anything, whatever its mean says.

When the headline metric is an error metric, a positive edge means more error and
therefore a worse model, and `folds_ahead` counts the folds where the model is
worse rather than better. The cell states which convention applies to the table
it prints.

In [ ]:
edge_table = pd.DataFrame()
if HAS_RUN and METRIC is not None and BASELINE is not None:
    edge_table = rep.edge_over_baseline(results, metric=METRIC, baseline=BASELINE)
    if edge_table.empty:
        print("Edge table is empty. The metric or the baseline model is absent from this run.")
    else:
        if LOWER_IS_BETTER:
            print("Metric %s: lower is better." % METRIC)
            print("A NEGATIVE mean_edge is an improvement over %s." % BASELINE)
            print("folds_ahead and win_rate count folds where the model has HIGHER error,")
            print("so for this metric a low win_rate is the good outcome.")
        else:
            print("Metric %s: higher is better." % METRIC)
            print("A POSITIVE mean_edge is an improvement over %s," % BASELINE)
            print("and win_rate is the share of folds the model wins.")
        display(edge_table)
else:
    print("No run loaded, edge table unavailable.")

## Seed spread

Models with a random component are trained several times per fold under different
seeds. The spread across those seeds is the noise floor of the experiment: if a
model's advantage over the baseline is smaller than its own seed to seed standard
deviation, the advantage has not been demonstrated and reporting it as a finding
would be reporting a coin flip.

Deterministic models such as the baselines, ARIMA and the GARCH family carry no
seed, so they are absent from this table by construction. An empty table means no
seeded model ran in this configuration, which is a fact about the run rather than
a failure of the analysis.

In [ ]:
spread_table = pd.DataFrame()
if HAS_RUN and METRIC is not None:
    spread_table = rep.seed_spread(results, metric=METRIC)
    if spread_table.empty:
        seeded = results["seed"].notna().sum()
        print("No seed spread to report.")
        print("Rows carrying a seed: %d of %d." % (seeded, len(results)))
        print("Every model in this run is deterministic, so repeated runs would be identical")
        print("and there is no sampling noise from seeding to net out of the comparisons.")
    else:
        display(spread_table)
else:
    print("No run loaded, seed spread unavailable.")

## Ground truth for the prediction level breakdowns

The per asset and per regime tables compare stored predictions against the
realised target, so the target has to be rebuilt from the panel. `build_targets`
is the single definition used by the training pipeline as well, so this is the
same column the models were scored on rather than a reconstruction of it.

Regimes are labelled by `classify_regime` using the trailing return over the
window set in the `evaluate.regimes` block of the config. The label uses only
trailing data, so it is computable at the time and does not put future
information into the breakdown.

In [ ]:
panel = pd.DataFrame()
y_true = pd.Series(dtype=float)
regimes = pd.Series(dtype=object)

if HAS_RUN and not predictions.empty:
    panel = load_panel(ROOT / "data" / "interim" / "panel_1d.parquet")
    config = load_config(ROOT / "config" / "base.yaml")
    targets = build_targets(panel)

    if TARGET in targets.columns:
        y_true = targets[TARGET]
    else:
        print("Target %r is not produced by build_targets, breakdowns will be skipped." % TARGET)

    regime_config = config.get("evaluate", {}).get("regimes", {})
    regimes = rep.classify_regime(
        panel,
        lookback_days=regime_config.get("lookback_days", 90),
        bull_threshold=regime_config.get("bull_threshold", 0.10),
        bear_threshold=regime_config.get("bear_threshold", -0.10),
    )
    print("ground truth rows: %d" % y_true.notna().sum())
    print(regimes.value_counts().to_string())
else:
    print("No predictions available, breakdown inputs not built.")

## Per asset breakdown

Skill can exist on one asset and nowhere else. The least liquid name in a panel
is the most plausible place for an inefficiency to survive, and pooling across
assets would average that away against three series where nothing is happening.

For a classification run the table reports directional accuracy next to that
asset's own base rate, because accuracy without the base rate beside it is
unreadable: 56 percent accuracy on an asset that rose on 56 percent of days is
exactly no skill. For a regression run the accuracy comparison is meaningless, so
the cell falls back to per asset RMSE and MAE computed directly from the stored
predictions. That fallback is computed on whatever scale the predictions were
written in, which for the volatility target is log variance, so it will not
match the headline error in `results.csv` if the pipeline scored on a
transformed scale. Compare the fallback numbers with each other, not with the
summary table.

In [ ]:
def error_breakdown(frame, group_columns):
    """RMSE and MAE per group, for runs where the target is continuous."""
    # The MultiIndex is dropped first because 'asset' exists both as an index
    # level and as a column here, and groupby refuses to guess which is meant.
    work = frame.reset_index(drop=True)
    work["error"] = work["pred"] - work["y"]
    grouped = work.groupby(group_columns, observed=True)["error"]
    out = pd.DataFrame({
        "rmse": grouped.apply(lambda s: float(np.sqrt(np.mean(np.square(s))))),
        "mae": grouped.apply(lambda s: float(np.mean(np.abs(s)))),
        "n": grouped.size(),
    })
    return out.reset_index().round(4)


scored = pd.DataFrame()
if HAS_RUN and not predictions.empty and y_true.notna().any():
    scored = predictions.copy()
    scored["y"] = y_true.reindex(scored.index)
    scored = scored.dropna(subset=["y", "pred"])
    scored["asset"] = scored.index.get_level_values(schema.ASSET)
    scored["regime"] = regimes.reindex(scored.index)
    print("scored prediction rows: %d" % len(scored))
else:
    print("Nothing to score.")

In [ ]:
per_asset = pd.DataFrame()
if not scored.empty:
    if IS_CLASSIFICATION:
        per_asset = rep.per_asset_breakdown(predictions, y_true)
        display(per_asset)
    else:
        print("Regression run on target %r." % TARGET)
        print("Directional accuracy is not defined here, reporting error per asset instead.")
        per_asset = error_breakdown(scored, ["model", "asset"])
        display(per_asset.pivot(index="asset", columns="model", values="rmse"))
        display(per_asset)
else:
    print("No scored predictions, per asset breakdown unavailable.")

## Per regime breakdown

Regime dependence is the most common reason a model looks convincing on one
sample and fails on the next. A directional model that is only right in bull
windows has learned the sample's drift, not a forecast, and averaging over
regimes hides exactly that.

The `unknown` regime covers the first 90 days of each asset, where the trailing
return needed for the label does not exist yet.

In [ ]:
per_regime = pd.DataFrame()
if not scored.empty:
    if IS_CLASSIFICATION:
        per_regime = rep.regime_breakdown(predictions, y_true, regimes)
        display(per_regime)
        if not per_regime.empty:
            pivot = per_regime.pivot(index="regime", columns="model", values="accuracy")
            figure, axis = plt.subplots(figsize=(11, 4.2))
            pivot.plot(kind="bar", ax=axis, width=0.8)
            axis.axhline(0.5, color="black", linewidth=0.8)
            axis.set_ylabel("directional accuracy")
            axis.set_title("Directional accuracy by market regime")
            axis.legend(fontsize=8, ncol=3)
            figure.tight_layout()
            plt.show()
    else:
        print("Regression run on target %r, reporting error per regime instead." % TARGET)
        per_regime = error_breakdown(scored, ["model", "regime"])
        display(per_regime.pivot(index="regime", columns="model", values="rmse"))
else:
    print("No scored predictions, per regime breakdown unavailable.")

## Feature importance

Feature importance is written only by runs that include a model able to report
it, so `feature_importance.csv` is often absent and its absence is not an error.

Two things are shown when the file exists. The first is the top 20 features by
mean gain, which says what the model leaned on overall. The second is more
informative and less often reported: how far the rank of each top feature moves
between folds. A feature that is ranked second on one fold and fortieth on the
next is not a stable driver, and a story built on the averaged ranking would be
describing an artefact of the averaging. Rank instability is a finding in its own
right, and on noisy financial data it is the expected one.

In [ ]:
importance = pd.DataFrame()
importance_path = None
if HAS_RUN:
    importance_path = run_dir / "feature_importance.csv"
    if importance_path.exists():
        importance = pd.read_csv(importance_path)
        print("loaded %s with %d rows" % (importance_path, len(importance)))
        print("columns: %s" % importance.columns.tolist())
    else:
        print("No feature_importance.csv in %s." % run_dir)
        print("This run contains no model that reports feature importance, so the two")
        print("cells below have nothing to draw and will say so.")
else:
    print("No run loaded, feature importance unavailable.")

In [ ]:
def pick_column(frame, candidates):
    for name in candidates:
        if name in frame.columns:
            return name
    return None


top_features = pd.DataFrame()
per_fold_gain = pd.DataFrame()
feature_column = gain_column = fold_column = model_column = None

if not importance.empty:
    feature_column = pick_column(importance, ["feature", "name", "column"])
    gain_column = pick_column(
        importance, ["gain", "mean_gain", "importance", "value", "score", "weight"]
    )
    fold_column = pick_column(importance, ["fold", "fold_id"])
    model_column = pick_column(importance, ["model"])

if feature_column is None or gain_column is None:
    if not importance.empty:
        print("Could not identify the feature and gain columns in this file.")
        print("Columns present: %s" % importance.columns.tolist())
    else:
        print("No feature importance table, top 20 unavailable.")
else:
    subset = importance
    if model_column is not None and importance[model_column].nunique() > 1:
        owner = importance[model_column].value_counts().idxmax()
        subset = importance[importance[model_column] == owner]
        print("Several models report importance, showing %r." % owner)

    # A run trains the same model under several seeds, so the same feature and
    # fold appear more than once. Averaging over seeds first keeps the fold, not
    # the seed, as the unit of comparison.
    if fold_column is not None:
        per_fold_gain = (
            subset.groupby([fold_column, feature_column], observed=True)[gain_column]
            .mean()
            .reset_index()
        )
        source = per_fold_gain
        unit = "folds"
    else:
        source = subset
        unit = "rows"

    top_features = (
        source.groupby(feature_column, observed=True)[gain_column]
        .agg(mean_gain="mean", std_gain="std", n=("size"))
        .reset_index()
        .sort_values("mean_gain", ascending=False)
        .head(20)
        .round(4)
    )
    top_features = top_features.rename(columns={"n": "n_" + unit})
    display(top_features)

    figure, axis = plt.subplots(figsize=(10, 6.5))
    labels = top_features[feature_column].tolist()[::-1]
    axis.barh(range(len(labels)), top_features["mean_gain"].tolist()[::-1], color="#3b6ea5")
    axis.set_yticks(range(len(labels)), labels, fontsize=8)
    axis.set_xlabel("mean %s per fold" % gain_column)
    axis.set_title("Top 20 features by mean gain")
    figure.tight_layout()
    plt.show()

In [ ]:
if not top_features.empty and not per_fold_gain.empty:
    ranked = per_fold_gain.copy()
    ranked["rank"] = (
        ranked.groupby(fold_column, observed=True)[gain_column]
        .rank(ascending=False, method="min")
    )
    n_features = int(ranked.groupby(fold_column, observed=True)[feature_column].nunique().max())
    print("ranking %d features within each of %d folds"
          % (n_features, ranked[fold_column].nunique()))

    top_ten = top_features[feature_column].head(10).tolist()
    rank_wide = (
        ranked[ranked[feature_column].isin(top_ten)]
        .pivot_table(index=fold_column, columns=feature_column, values="rank", aggfunc="min")
        .sort_index()
    )

    figure, axes = plt.subplots(1, 2, figsize=(13, 5))
    for column in rank_wide.columns:
        axes[0].plot(rank_wide.index, rank_wide[column], "o-", markersize=3,
                     linewidth=1.0, label=column)
    axes[0].invert_yaxis()
    axes[0].set_xlabel("fold")
    axes[0].set_ylabel("importance rank, 1 is most important")
    axes[0].set_title("Rank of the top 10 features across folds")
    axes[0].legend(fontsize=7, ncol=2)

    movement = pd.DataFrame({
        "best_rank": rank_wide.min(),
        "worst_rank": rank_wide.max(),
        "rank_std": rank_wide.std(),
    })
    movement["rank_range"] = movement["worst_rank"] - movement["best_rank"]
    movement = movement.sort_values("rank_range", ascending=False)

    axes[1].barh(range(len(movement)), movement["rank_range"].to_numpy(), color="#e08214")
    axes[1].set_yticks(range(len(movement)), movement.index.tolist(), fontsize=8)
    axes[1].invert_yaxis()
    axes[1].set_xlabel("best to worst rank spread across folds")
    axes[1].set_title("How far each top feature moves")
    figure.tight_layout()
    plt.show()

    display(movement.round(2))
elif not top_features.empty:
    print("The importance file has no fold column, so rank movement cannot be measured.")
else:
    print("No feature importance table, rank stability plot unavailable.")

## Reading the run as a whole

The questions to take away from the tables above, in the order they should be
asked:

1. Does any model beat the baseline on the headline metric by more than the fold
   to fold standard deviation of that difference. If not, the ranking in the
   summary table is noise.
2. Does it win on a clear majority of folds. A mean edge carried by two folds out
   of twenty is one market episode, not a forecast.
3. Is the edge present in more than one asset and more than one regime. An edge
   confined to a single asset in bull windows is a description of the sample.
4. Is the edge larger than the seed to seed spread of the same model, where the
   model has a seed at all.

A run that fails these is still a result. On daily crypto direction the expected
finding is that the baselines are close to unbeatable, and reporting that
honestly is the point of structuring the evaluation this way.